# Advanced Wi-Fi Track Data Clean

This method re-clean data according to track repeat frequency ( caused by unstable signal appears among multiple trackers) based on given basic cleaned data.

Generate **virtual wifi tracker** after cleaning.(Virtual wifi trackers' id will be under zero)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

## Read Data

In [10]:
df_read = pd.read_csv(os.getcwd()+'/cleaned_data/dacang/dacang_cleaned_data_basic.csv')
df = df_read.copy()
df_read

,a,r,t,m
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98"
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53"
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106"
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53"
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106"
...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144"
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144"
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144"
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144"


In [11]:
#label each mac with date
pbar = tqdm(total=len(df))
df['oriMac'] = df['m']
df.t = pd.to_datetime(df.t)
df['calender'] = df.t.apply(lambda x : str(x.month)+'.'+str(x.day))
for index,row in df.iterrows():
    df.loc[index,'m'] = str(row['calender'])+'-'+row.m
    pbar.update(1)
pbar.close()
df

100%|██████████| 1916101/1916101 [01:38<00:00, 19509.40it/s]


,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [12]:
len(df.m.unique())

31798

In [13]:
#删除打上标签后只出现过一次的mac
df = df[df.duplicated('m',keep=False)]
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 保存或读取已打上date标签的dataframe

In [67]:
#df.to_csv(os.getcwd()+"/cleaned_data/dacang/temp/dacang_cleaned_data_dataLabeled.csv",index=False)
df = pd.read_csv(os.getcwd()+"/cleaned_data/dacang/temp/dacang_cleaned_data_dataLabeled.csv")
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 以天为单位清理异常mac

In [59]:
#获取每个mac一天中经过的探针数：a_count
count_list = df.groupby('m').a.value_counts()
df_count = count_list.to_frame().reset_index()
a_count = df_count.m.value_counts()
a_count

m
7.19-64,140,31,142,167,149     16
7.24-48,231,188,153,110,173    15
7.27-64,140,31,142,167,149     14
7.14-88,198,240,41,58,135      14
7.19-88,198,240,41,58,135      14
                               ..
7.15-16,148,53,197,58,225       1
7.15-16,74,38,133,29,213        1
7.28-240,103,40,199,90,171      1
7.28-240,103,40,135,90,171      1
7.25-164,8,250,121,194,19       1
Name: count, Length: 15615, dtype: int64

In [60]:
#获取每个mac一天中的数据量:s_count
count_list = df.m.value_counts()
count_list

m
7.10-48,74,38,155,214,98        22368
7.9-48,74,38,155,214,98         14662
7.7-48,74,38,155,214,98         11741
7.8-48,74,38,155,214,98         11646
7.11-48,74,38,155,214,98        10947
                                ...  
7.28-204,129,106,183,42,73          2
7.19-64,204,31,142,167,149          2
7.28-172,254,224,205,171,204        2
7.19-112,138,9,131,173,199          2
7.14-120,251,20,50,26,70            2
Name: count, Length: 15615, dtype: int64

In [68]:
#获取每个mac一天中的持续时间
df.t = pd.to_datetime(df.t)
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [69]:
mac_list = df.m.unique()
len(mac_list)

15615

In [83]:
def GetDuration(df_now):
    time = df_now.iloc[len(df_now)-1].t - df_now.iloc[0].t
    time = time.components.hours+(time.components.minutes/60)
    return round(time,2)

macDur_dict = {}

for mac in tqdm(mac_list):
    df_now = df[df.m == mac]
    time = GetDuration(df_now)
    macDur_dict.update({mac:time})

macDur_dict

 17%|█▋        | 2627/15615 [03:14<16:01, 13.50it/s]


KeyboardInterrupt: 

In [30]:
#获得忽略数据次数的mac-a列表
count_list = df.groupby(['m']).a.value_counts()
df_count = count_list.to_frame().reset_index()
#获得一天内出现在超过4个位置的mac
counts = df_count.m.value_counts()
mac_list = counts[counts >= 4].index
#筛选出满足条件的mac对应的数据
df = df[df.m.isin(mac_list)]
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
6,22,-43,2022-07-05 23:49:47,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
8,22,-43,2022-07-05 23:50:00,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
10,22,-35,2022-07-05 23:50:00,"7.5-20,178,229,137,47,142","20,178,229,137,47,142",7.5
12,22,-42,2022-07-05 23:50:13,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
...,...,...,...,...,...,...
1916076,32,-34,2022-08-07 23:51:18,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7
1916081,32,-42,2022-08-07 23:51:44,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7
1916082,32,-27,2022-08-07 23:51:57,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7
1916085,32,-36,2022-08-07 23:52:49,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7


In [50]:
count_list = df.m.value_counts()
fig = px.bar(count_list)
fig.show()